# Real data (Forstmann et al., 2008) with TINN and PyDDM

Fits the speed / neutral / accuracy data of Forstmann et al. (2008) (`data/FDBNCRW2008.RData`: 19 participants,
15,818 trials) with one of four models in which only the bound differs between the three instructions:
`ddm3`, `ou3`, `collapse3` or `urgency3`. Set `MODEL` and run all cells; each participant is fitted separately.

Results go to `results/<MODEL>_subject/` next to this notebook (`tinn.csv`, `pyddm.csv`) and the tables
and density figure to its `paper/` folder. Participants already fitted are skipped, so a stopped run
continues where it stopped; keep the number of workers the same between runs.

## 1. Settings

In [ ]:
MODEL = "ou3"             # "ddm3", "ou3", "collapse3", "urgency3"
FIT_MODE = "subject"      # "subject": each participant separately; "pooled": one fit to all data

RUN_PYDDM, RUN_TINN = True, True
PYDDM_WORKERS = 2         # keep equal to the earlier run (Colab used 2) when resuming
TINN_WARM, TINN_ITERS = 2000, 6000
TINN_DEVICE = "auto"      # "auto" = GPU if there is one
TINN_WORKERS = 3          # keep equal to the earlier run (Colab used 3) when resuming
TINN_WARM_START = False   # True = fast mode (pooled fit first, then each subject starts from it and stops early);
                          # keep False to match subjects already fitted in Colab with the standard settings
TINN_SUBJ_WARM, TINN_SUBJ_ITERS, TINN_TOL = 200, 6000, 1e-3

## 2. Results folder and helpers

In [ ]:
import os, sys, glob, subprocess, threading
import numpy as np, pandas as pd

CODE = os.path.abspath(".")                 # this folder (3_data_fitting)
assert os.path.exists(os.path.join(CODE, "fit_real.py")), "open this notebook from 3_data_fitting/"
RAW = os.path.join(CODE, "data", "FDBNCRW2008.RData")
R = os.path.join(CODE, "results", f"{MODEL}_{FIT_MODE}")
os.makedirs(R, exist_ok=True)
ENV = dict(os.environ, OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
print("code:", CODE)
print("results folder:", R)


def _show(p):
    for line in p.stdout:
        if "Warning" not in line and not line.startswith((" ", "Info")):
            print(line, end="", flush=True)


def run_parallel(method, workers, extra=()):
    """fits the subjects in `workers` processes at once (each takes a fixed slice of the subjects),
    then merges the parts into <R>/<method>.csv.  Subjects already in a part file are skipped."""
    n = len(SUBJECTS)
    step = -(-n // workers)
    procs = []
    for w in range(workers):
        first, last = w * step, min((w + 1) * step, n)
        if first < last:
            procs.append(subprocess.Popen(
                [sys.executable, "-u", "fit_real.py", "--data", DATA_CSV, "--model", MODEL, "--method", method,
                 "--out", os.path.join(R, f"{method}_part{w}.csv"), "--first", str(first), "--last", str(last),
                 *map(str, extra)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=ENV))
    threads = [threading.Thread(target=_show, args=(p,)) for p in procs]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    if any(p.wait() for p in procs):
        raise RuntimeError(f"{method} failed (see the messages above)")
    parts = [pd.read_csv(f) for f in sorted(glob.glob(os.path.join(R, f"{method}_part*.csv")))]
    df = pd.concat(parts).drop_duplicates("subj")
    df.to_csv(os.path.join(R, f"{method}.csv"), index=False)
    print(f"{method}: {len(df)} subjects -> {R}")
    return df

## 3. Data

Correct response = upper bound (`choice = 1`). Conditions: 0 speed, 1 neutral, 2 accuracy. If `data.csv` is already in the results folder, it is used.

In [ ]:
DATA_CSV = os.path.join(R, "data.csv")
if not os.path.exists(DATA_CSV):
    import pyreadr
    raw = pyreadr.read_r(RAW)["data"]
    data = pd.DataFrame({"subj": raw["subj"].astype(str), "rt": raw["RT"].astype(float),
                         "choice": raw["correct"].astype(int),
                         "cond": raw["instruction"].astype(str).map({"speed": 0, "neutral": 1, "accuracy": 2})}).dropna()
    if FIT_MODE == "pooled":
        data["subj"] = "all"
    data.to_csv(DATA_CSV, index=False)
data = pd.read_csv(DATA_CSV)
SUBJECTS = list(dict.fromkeys(data["subj"].astype(str)))
POOLED_CSV = os.path.join(R, "data_pooled.csv")
data.assign(subj="all").to_csv(POOLED_CSV, index=False)
print(f"{len(data)} trials, {len(SUBJECTS)} subject(s)")
print(data.groupby("cond").agg(trials=("rt", "size"), mean_rt=("rt", "mean"), accuracy=("choice", "mean")).round(3))

## 4. PyDDM

In [ ]:
if RUN_PYDDM:
    pyddm_df = run_parallel("pyddm", PYDDM_WORKERS)

## 5. TINN

In [ ]:
if RUN_TINN:
    import torch
    print("TINN on", torch.cuda.get_device_name() if torch.cuda.is_available() and TINN_DEVICE != "cpu" else "CPU")
    if TINN_WARM_START and FIT_MODE == "subject":
        pooled = os.path.join(R, "tinn_pooled.pt")
        if not os.path.exists(pooled):
            print("pooled TINN fit to all trials ...", flush=True)
            subprocess.run([sys.executable, "-u", "fit_real.py", "--data", POOLED_CSV, "--model", MODEL, "--method", "tinn",
                            "--out", os.path.join(R, "tinn_pooled.csv"), "--save", pooled, "--warm", str(TINN_WARM),
                            "--iters", str(TINN_ITERS), "--device", TINN_DEVICE], check=True, env=ENV)
        tinn_df = run_parallel("tinn", TINN_WORKERS, ["--init", pooled, "--warm", TINN_SUBJ_WARM, "--iters", TINN_SUBJ_ITERS,
                                                      "--tol", TINN_TOL, "--device", TINN_DEVICE])
    else:
        tinn_df = run_parallel("tinn", TINN_WORKERS, ["--warm", TINN_WARM, "--iters", TINN_ITERS, "--device", TINN_DEVICE])

## 6. Tables and densities

Written to the `paper` folder inside the results folder: `table_tinn.tex`, `table_pyddm.tex`, `table_fit.csv`, `densities.pdf`.

In [ ]:
r = subprocess.run([sys.executable, "real_report.py", "--model", MODEL, "--results", R, "--data", DATA_CSV],
                   capture_output=True, text=True)
print("\n".join(l for l in (r.stdout + r.stderr).splitlines() if "Warning" not in l))

from IPython.display import Image, display
display(Image(os.path.join(R, "paper", "densities.png"), width=1000))